# 🚢 Titanic Survival Prediction
## Machine Learning + Deep Learning + Streamlit

This notebook follows the supplied assignment brief: load the Titanic dataset, handle missing values, train five Machine Learning models, train ANN and 1D CNN models when TensorFlow is available, compare metrics, and save the best Machine Learning pipeline.

**Dataset:** 891 passengers × 12 columns.  
**Primary comparison metric:** F1 Score, with Accuracy and ROC-AUC as supporting metrics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
import joblib

DATA_PATH = "../data/Titanic-Dataset.csv"
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
print("Shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())
print("\nTarget distribution:")
print(df["Survived"].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=df, x="Survived", ax=ax)
ax.set_title("Survival Distribution")
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=df, x="Sex", hue="Survived", ax=ax)
ax.set_title("Survival by Gender")
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=df, x="Pclass", hue="Survived", ax=ax)
ax.set_title("Survival by Passenger Class")
plt.show()

In [ ]:
features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
X = df[features]
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

numeric_features = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Sex", "Embarked"]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

In [ ]:
ml_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=5),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "XGBoost": XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="logloss", random_state=42
    )
}

ml_results = []
trained_models = {}

for name, estimator in ml_models.items():
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    ml_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })
    trained_models[name] = model

ml_results_df = pd.DataFrame(ml_results).sort_values("F1 Score", ascending=False)
ml_results_df

In [ ]:
ax = ml_results_df.set_index("Model")[["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"]].plot(
    kind="bar", figsize=(12, 6)
)
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Machine Learning Model Comparison")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## Deep Learning Models: ANN and 1D CNN

The supplied project brief specifies TensorFlow/Keras for the ANN and 1D CNN. The following cells are intentionally guarded so the notebook does not crash in an environment where TensorFlow is not installed. On Google Colab, TensorFlow is normally available; otherwise install the package listed in `requirements-full.txt` and rerun these cells.

In [ ]:
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Dense, Dropout, Conv1D, MaxPooling1D, Flatten
    TENSORFLOW_AVAILABLE = True
    print("TensorFlow version:", tf.__version__)
except ImportError:
    TENSORFLOW_AVAILABLE = False
    print("TensorFlow is not installed. Install it from requirements-full.txt to run ANN and 1D CNN.")

In [ ]:
dl_numeric = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
dl_categorical = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
dl_preprocessor = ColumnTransformer([
    ("num", dl_numeric, numeric_features),
    ("cat", dl_categorical, categorical_features)
])

X_train_dl = dl_preprocessor.fit_transform(X_train).toarray()
X_test_dl = dl_preprocessor.transform(X_test).toarray()

dl_results = []

if TENSORFLOW_AVAILABLE:
    ann_model = Sequential([
        Dense(64, activation="relu", input_shape=(X_train_dl.shape[1],)),
        Dropout(0.30),
        Dense(32, activation="relu"),
        Dropout(0.20),
        Dense(16, activation="relu"),
        Dense(1, activation="sigmoid")
    ])
    ann_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    ann_model.fit(
        X_train_dl, y_train,
        validation_split=0.20,
        epochs=50,
        batch_size=32,
        verbose=0
    )
    ann_prob = ann_model.predict(X_test_dl, verbose=0).ravel()
    ann_pred = (ann_prob >= 0.5).astype(int)
    dl_results.append({
        "Model": "ANN",
        "Accuracy": accuracy_score(y_test, ann_pred),
        "Precision": precision_score(y_test, ann_pred, zero_division=0),
        "Recall": recall_score(y_test, ann_pred, zero_division=0),
        "F1 Score": f1_score(y_test, ann_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, ann_prob)
    })

    X_train_cnn = X_train_dl.reshape(X_train_dl.shape[0], X_train_dl.shape[1], 1)
    X_test_cnn = X_test_dl.reshape(X_test_dl.shape[0], X_test_dl.shape[1], 1)

    cnn_model = Sequential([
        Conv1D(32, kernel_size=3, activation="relu", input_shape=(X_train_cnn.shape[1], 1)),
        MaxPooling1D(pool_size=2),
        Conv1D(64, kernel_size=3, activation="relu"),
        Flatten(),
        Dense(32, activation="relu"),
        Dropout(0.30),
        Dense(1, activation="sigmoid")
    ])
    cnn_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    cnn_model.fit(
        X_train_cnn, y_train,
        validation_split=0.20,
        epochs=50,
        batch_size=32,
        verbose=0
    )
    cnn_prob = cnn_model.predict(X_test_cnn, verbose=0).ravel()
    cnn_pred = (cnn_prob >= 0.5).astype(int)
    dl_results.append({
        "Model": "1D CNN",
        "Accuracy": accuracy_score(y_test, cnn_pred),
        "Precision": precision_score(y_test, cnn_pred, zero_division=0),
        "Recall": recall_score(y_test, cnn_pred, zero_division=0),
        "F1 Score": f1_score(y_test, cnn_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, cnn_prob)
    })

dl_results_df = pd.DataFrame(dl_results)
dl_results_df

In [ ]:
all_results = pd.concat([ml_results_df, dl_results_df], ignore_index=True)
all_results = all_results.sort_values("F1 Score", ascending=False)
all_results

In [ ]:
best_model_name = ml_results_df.iloc[0]["Model"]
best_ml_model = trained_models[best_model_name]

joblib.dump(best_ml_model, "../models/titanic_best_model.pkl")
ml_results_df.to_csv("../results/model_results.csv", index=False)

print("Best Machine Learning model by F1 Score:", best_model_name)
print("Saved: ../models/titanic_best_model.pkl")

In [ ]:
y_pred = best_ml_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

print(classification_report(y_test, y_pred, zero_division=0))

## Final Workflow

**Data → Preprocessing → Machine Learning → Deep Learning → Evaluation → Best Machine Learning Model → Streamlit**

The Streamlit app uses the saved complete scikit-learn pipeline, so imputation, one-hot encoding, scaling, and prediction are applied consistently to new passenger input.